# LLM-only baseline (no ontology)

Separate from the SNOMED / symptom-tree pipeline.

**Inputs (read-only):** `patient_records/` redacted notes + `clinical_context.txt` + `admission_history.json` + GT for scoring.

**Outputs (this folder only):** `llm_only_baseline/`
- per admission: DiffDx + LLM ICD + `accuracy/`
- cohort: `cohort_accuracy.txt`, CSVs, QE cache

Does **not** write into `patient_records/*/accuracy` or `data/stage_10_evaluation/`.

**Metrics:** DiffDx vs physician Discharge Diagnosis; ICD vs billed primary ICD.

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

from pipeline import EXPORT_DIR, print_pipeline_banner
from llm_only_baseline import (
    LLM_ONLY_COHORT_TXT,
    LLM_ONLY_DIR,
    LLM_ONLY_DIFFDX_CSV,
    LLM_ONLY_ICD_CSV,
    LLM_ONLY_SUMMARY,
    run_llm_only_cohort,
)

print_pipeline_banner()
print(f"Source notes : {EXPORT_DIR}")
print(f"LLM-only out : {LLM_ONLY_DIR}")
print("(Ontology pipeline folders are not modified.)")

Pipeline mode : FULL (15 patients)
LLM provider  : OpenRouter (anthropic/claude-opus-5, ZDR on)
Qwen pair     : Local equivalent: ollama pull qwen2.5:7b
Admissions/patient (min): 2
Data dir      : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data
Export dir    : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/patient_records
OpenRouter ZDR : enabled (provider.zdr=true on every request)
Source notes : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/patient_records
LLM-only out : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/llm_only_baseline
(Ontology pipeline folders are not modified.)


## Run DiffDx + ICD + evaluation

- `skip_if_done=True` resumes from `llm_only_baseline/checkpoint.json`
- Set `skip_if_done=False` to regenerate all admissions
- `force_llm_qe_refresh=True` rebuilds LLM-only QE cache under this folder

In [3]:
payload = run_llm_only_cohort(
    export_dir=EXPORT_DIR,
    skip_if_done=True,
    evaluate=True,
    force_llm_qe_refresh=True,
)
print(f"\nSource admissions : {payload['n_source']}")
print(f"Run records       : {payload['n_run']}")
print(f"Evaluated         : {payload['n_evaluated']}")
print(f"Output dir        : {payload['output_dir']}")
print(f"Cohort accuracy   : {LLM_ONLY_COHORT_TXT}")
print(f"DiffDx CSV        : {LLM_ONLY_DIFFDX_CSV}")
print(f"ICD CSV           : {LLM_ONLY_ICD_CSV}")
print(f"Summary JSON      : {LLM_ONLY_SUMMARY}")

LLM-only baseline → /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/llm_only_baseline
Source admissions : 15 under /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/patient_records
Model             : anthropic/claude-opus-5  temp=0.4
[1/15] patient_10361982/hadm_24286431 …
  → ok  /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/llm_only_baseline/patient_10361982/hadm_24286431
[2/15] patient_10426859/hadm_29908281 …
  → ok  /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/llm_only_baseline/patient_10426859/hadm_29908281
[3/15] patient_10458324/hadm_21744342 …
  Note: repaired truncated JSON from LLM output
  → ok  /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/llm_only_baseline/patient_10458324/hadm_21744342
[4/15] patient_11251337/hadm_29568708 …
  → ok  /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/llm_only_baseline/patient_11251337/hadm_29568708
[5/15] patient_11474876/hadm_29672491 …
  → ok  /Users/narenkhatwani/GitHub/ai-

In [4]:
from pathlib import Path

txt = Path(LLM_ONLY_COHORT_TXT)
if txt.exists():
    print(txt.read_text(encoding="utf-8")[:4000])
else:
    print("No cohort_accuracy.txt yet — run the cell above.")

STAGE 10 — ACCURACY (LLM query expansion)

Admissions scored : 15
With Stage 9      : 15

DIFFERENTIAL DIAGNOSIS vs physician Discharge Diagnosis (note)
  Any-rank match               15 / 15  (100.0%)
  Rank-1 only                  13 / 15  (86.7%)
  Lower-rank only              2 / 15

LINKED ICD vs billed primary ICD (from matched DiffDx rank)
  ICD match (exact or LLM)     10 / 15  (66.7%)
  ICD exact code               5 / 15  (33.3%)
  ICD 3-char family            9 / 15  (60.0%)
  Diagnosis YES, ICD NO        5

→ diffdx_match.csv, icd_match.csv

PER-ADMISSION SUMMARY
---------------------

Dx   R1   ICD  Rank  Patient      Matched diagnosis                    GT (discharge dx)
--------------------------------------------------------------------------------------------------------------
YES  YES  YES  1     10361982     Postprocedural hemorrhage from ...   Heavy vaginal bleeding after total lapa...
YES  YES  YES  1     10426859     Recurrent paroxysmal supraventr...   supraventr